# MS1MV3 → FaceXFormer Encoder Feature Extraction

- HuggingFace에서 레포 통째로 다운로드
- 4 GPU DataParallel, no grad, fp16
- **model.forward() 우회** — backbone → linear_c → linear_fuse 직접 호출
- 라벨 순서 보장 (shuffle=False)
- FaceXFormer 전처리: 224x224 Bicubic, ImageNet norm

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

import sys
import numbers
import time

import mxnet as mx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 0. Config

In [ ]:
# TODO: local paths in this cell were redacted to ""; set them before running.
# ============ 여기만 수정 ============
REC_DIR      = ""   # MS1MV3 RecordIO 경로
OUTPUT_PATH  = ""      # feature 저장 경로
LABELS_PATH  = ""                 # 라벨 저장/비교 경로
BATCH_SIZE   = 256
NUM_WORKERS  = 8
# ======================================

## 1. FaceXFormer 다운로드 & 로드

In [ ]:
from huggingface_hub import snapshot_download

facex_dir = snapshot_download(
    repo_id="kartiknarayan/facexformer",
    local_dir="./facexformer_hf",
)
print(f"Downloaded to: {facex_dir}")

if facex_dir not in sys.path:
    sys.path.insert(0, facex_dir)

In [ ]:
from network import FaceXFormer

ckpt_path = os.path.join(facex_dir, "ckpts", "model.pt")
model = FaceXFormer()
checkpoint = torch.load(ckpt_path, map_location="cpu")

if isinstance(checkpoint, dict) and 'state_dict_backbone' in checkpoint:
    model.load_state_dict(checkpoint['state_dict_backbone'], strict=False)
    print("Loaded with key 'state_dict_backbone'")
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'], strict=False)
    print("Loaded with key 'state_dict'")
else:
    model.load_state_dict(checkpoint, strict=False)
    print("Loaded directly")

for p in model.parameters():
    p.requires_grad_(False)
model.eval()
print("FaceXFormer loaded & frozen")

## 2. Backbone 구조 디버깅

`model.forward()`는 backbone hook이 torchvision 버전에 따라 깨질 수 있으므로,
backbone 내부 모듈 이름을 직접 확인한 후 우회 호출한다.

In [ ]:
print("[Top-level modules]")
for name, child in model.named_children():
    n_params = sum(p.numel() for p in child.parameters())
    print(f"  {name}: {type(child).__name__} ({n_params/1e6:.1f}M)")

print(f"\n[model.target_layer_names] = {model.target_layer_names}")

print("\n[backbone named_modules (depth<=2)]")
for name, mod in model.backbone.named_modules():
    if name.count('.') <= 1 and name != '':
        print(f"  '{name}': {type(mod).__name__}")

## 3. Encoder Feature Extractor (forward 우회)

model.forward()를 쓰지 않고, backbone → hook으로 multi-scale feature 수집 → linear_c → linear_fuse 직접 호출.

```
backbone(x)
  ├─ hook on stage 1 → feat[0]  (128-ch)
  ├─ hook on stage 2 → feat[1]  (256-ch)
  ├─ hook on stage 3 → feat[2]  (512-ch)
  └─ hook on stage 4 → feat[3]  (1024-ch)
         ↓
    linear_c[i](feat[i]) → 256-ch each
         ↓
    concat → linear_fuse → [B, 256, H, W]
         ↓
    avg pool → [B, 256]  ← 이게 encoder feature
```

In [ ]:
class FaceXEncoderExtractor(nn.Module):
    """
    FaceXFormer encoder feature 추출 (hook-free, DataParallel 호환).
    backbone Sequential을 수동으로 순회하면서 target stage의 output을 캡쳐.
    """
    def __init__(self, facex_model):
        super().__init__()
        self.linear_c = facex_model.linear_c
        self.linear_fuse = facex_model.linear_fuse
        
        backbone = facex_model.backbone
        children = list(backbone.children())
        self.features_seq = children[0]
        
        # target stage 인덱스: dummy forward로 shape 확인
        self.stage_indices = []
        target_channels = {128, 256, 512, 1024}
        
        print("backbone features sub-modules:")
        for i, (name, mod) in enumerate(self.features_seq.named_children()):
            print(f"  [{i}] {name}: {type(mod).__name__}")
        
        with torch.no_grad():
            x = torch.randn(1, 3, 224, 224)
            found_channels = []
            for i, mod in enumerate(self.features_seq.children()):
                x = mod(x)
                shape = list(x.shape)
                print(f"  [{i}] output shape: {shape}")
                # Swin stage output: [B, H, W, C] — C는 마지막 dim
                if x.dim() == 4:
                    c = shape[-1]
                    if c in target_channels and c not in found_channels:
                        self.stage_indices.append(i)
                        found_channels.append(c)
                        print(f"      ★ matched channel={c}")
        
        print(f"\nTarget stage indices: {self.stage_indices}")
        print(f"Matched channels: {found_channels}")
        assert len(self.stage_indices) == 4, (
            f"Expected 4 stages, found {len(self.stage_indices)}. "
            f"위 output shape을 확인하세요."
        )
    
    def forward(self, x):
        B = x.shape[0]
        
        multi_scale_features = []
        h = x
        for i, mod in enumerate(self.features_seq.children()):
            h = mod(h)
            if i in self.stage_indices:
                # Swin: [B, H, W, C] → [B, C, H, W]
                multi_scale_features.append(h.permute(0, 3, 1, 2).contiguous())
        
        target_size = multi_scale_features[0].shape[2:]
        
        all_hidden_states = []
        for feat, mlp in zip(multi_scale_features, self.linear_c):
            h_s, w_s = feat.shape[2], feat.shape[3]
            out = mlp(feat)                           # [B, H*W, 256]
            out = out.permute(0, 2, 1)                # [B, 256, H*W]
            out = out.reshape(B, -1, h_s, w_s)        # [B, 256, H, W]
            out = F.interpolate(
                out, size=target_size,
                mode='bilinear', align_corners=False
            )
            all_hidden_states.append(out)
        
        fused = self.linear_fuse(torch.cat(all_hidden_states[::-1], dim=1))
        out = fused.mean(dim=[2, 3])
        
        return out.float()


encoder = FaceXEncoderExtractor(model)
print(f"\nEncoder ready.")

In [ ]:
# feature shape 확인
device = torch.device("cuda:0")
encoder_test = encoder.to(device)

dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad(), torch.cuda.amp.autocast():
    feat = encoder_test(dummy)

FEAT_DIM = feat.shape[1]
print(f"Encoder feature shape: {feat.shape}")   # [2, 256]
print(f"FEAT_DIM = {FEAT_DIM}")

encoder_test = encoder_test.cpu()
torch.cuda.empty_cache()

## 4. MS1MV3 Dataset (FaceXFormer 전처리)

In [ ]:
class MXFaceForFaceX(Dataset):
    """
    MS1MV3 RecordIO → FaceXFormer 전처리.

    주의 — Face-Recog 학습과 전처리가 완전히 다름!
      Face-Recog:   112x112, Normalize(0.5, 0.5)                → [-1, 1]
      FaceXFormer:  224x224 Bicubic, ImageNet norm               → ~[-2.1, 2.6]

    Inference → RandomHorizontalFlip 없음.
    """
    def __init__(self, root_dir):
        super().__init__()
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])
        path_imgrec = os.path.join(root_dir, 'train.rec')
        path_imgidx = os.path.join(root_dir, 'train.idx')
        self.imgrec = mx.recordio.MXIndexedRecordIO(path_imgidx, path_imgrec, 'r')
        s = self.imgrec.read_idx(0)
        header, _ = mx.recordio.unpack(s)
        if header.flag > 0:
            self.header0 = (int(header.label[0]), int(header.label[1]))
            self.imgidx = np.array(range(1, int(header.label[0])))
        else:
            self.imgidx = np.array(list(self.imgrec.keys))

    def __getitem__(self, index):
        idx = self.imgidx[index]
        s = self.imgrec.read_idx(idx)
        header, img = mx.recordio.unpack(s)
        label = header.label
        if isinstance(label, numbers.Number):
            label = int(label)
        elif isinstance(label, (list, tuple, np.ndarray)):
            label = int(label[0])
        elif isinstance(label, np.generic):
            label = int(label.item())
        else:
            raise TypeError(f"Unexpected label type: {type(label)}")
        sample = mx.image.imdecode(img).asnumpy()  # (H, W, 3) uint8 RGB
        sample = self.transform(sample)
        return sample, label, index

    def __len__(self):
        return len(self.imgidx)


dataset = MXFaceForFaceX(REC_DIR)
print(f"Dataset size: {len(dataset)}")

## 5. Transform 검증

In [ ]:
img, label, idx = dataset[0]

print(f"Shape: {img.shape}")            # (3, 224, 224)
print(f"Dtype: {img.dtype}")            # float32
print(f"Range: [{img.min():.3f}, {img.max():.3f}]")
print(f"Label: {label}, Index: {idx}")

assert img.shape == (3, 224, 224), f"Shape mismatch: {img.shape}"
assert img.min() < -1.0, f"Min {img.min():.3f} — ImageNet norm이 적용 안 됨"
assert img.max() > 1.0,  f"Max {img.max():.3f} — ImageNet norm이 적용 안 됨"

for ch, name in enumerate(["R", "G", "B"]):
    print(f"  {name} mean={img[ch].mean():.3f}")

print("\nTransform 검증 통과!")

## 6. Feature 추출 (4 GPU)

encoder는 image만 받음 — dummy labels/tasks 불필요.

In [ ]:
device = torch.device("cuda:0")
encoder_dp = nn.DataParallel(encoder.to(device))
encoder_dp.eval()
print(f"DataParallel on {torch.cuda.device_count()} GPUs")

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # 순서 보장
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

In [ ]:
N = len(dataset)
all_feats = torch.zeros(N, FEAT_DIM, dtype=torch.float32)
all_labels = torch.zeros(N, dtype=torch.long)
filled = torch.zeros(N, dtype=torch.bool)

t0 = time.time()

with torch.no_grad():
    for batch_idx, (imgs, labels, indices) in enumerate(tqdm(dataloader, desc="Extracting")):
        imgs = imgs.to(device, non_blocking=True)

        with torch.cuda.amp.autocast():
            feat = encoder_dp(imgs)   # [B, FEAT_DIM]

        feat = feat.float().cpu()
        indices = indices.long()

        all_feats[indices] = feat
        all_labels[indices] = labels
        filled[indices] = True

        if batch_idx % 500 == 0 and batch_idx > 0:
            elapsed = time.time() - t0
            speed = (batch_idx * BATCH_SIZE) / elapsed
            print(f"  [{batch_idx}/{len(dataloader)}] {speed:.0f} img/s | filled={filled.sum().item()}/{N}")

total_time = time.time() - t0
print(f"\nDone: {total_time:.1f}s ({N / total_time:.0f} img/s)")
print(f"Filled: {filled.sum().item()}/{N}")
assert filled.all(), f"Missing {(~filled).sum().item()} samples!"

## 7. 라벨 순서 검증 & 저장

In [ ]:
unique, counts = all_labels.unique(return_counts=True)
print(f"Total samples: {len(all_labels)}")
print(f"Unique classes: {len(unique)}")
print(f"Samples/class: min={counts.min().item()}, max={counts.max().item()}, avg={counts.float().mean():.1f}")

if os.path.exists(LABELS_PATH):
    ref = torch.load(LABELS_PATH, map_location="cpu")
    M = min(len(ref), len(all_labels))
    match = (ref[:M] == all_labels[:M]).sum().item()
    print(f"\n기존 {LABELS_PATH}와 비교: {match}/{M} 일치 ({match/M*100:.2f}%)")
    if match != M:
        mismatch = (ref[:M] != all_labels[:M]).nonzero(as_tuple=True)[0][:5]
        for i in mismatch:
            print(f"  idx={i.item()}: ref={ref[i].item()} vs extracted={all_labels[i].item()}")
else:
    print(f"\n{LABELS_PATH} 없음 — 새로 저장합니다.")
    torch.save(all_labels, LABELS_PATH)
    print(f"Saved labels to {LABELS_PATH}")

In [ ]:
print(f"Features shape: {all_feats.shape}")
print(f"Features dtype: {all_feats.dtype}")
print(f"File size: ~{all_feats.element_size() * all_feats.nelement() / 1e9:.2f} GB")

torch.save(all_feats, OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

In [ ]:
# 저장 검증
loaded = torch.load(OUTPUT_PATH, map_location="cpu")
assert loaded.shape == all_feats.shape
assert torch.equal(loaded, all_feats)
print(f"저장 검증 통과: {loaded.shape}")
del loaded